In [1]:
import os
import shutil
import subprocess
from pyspark.sql import SparkSession

In [2]:
# os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/BIN:{os.environ.get('PATH', '')}"
# os.environ["PATH"]

In [3]:
# os.environ.get("JAVA_HOME")
# shutil.which("java")
# subprocess.run(["java", "-version"], capture_output=True, text=True).stderr

In [4]:
JARS_URLS = ",".join([
    "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar",
    "https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar",
])

In [5]:
DRIVER_HOST = "host.docker.internal"
DRIVER_PORT = "4042"
BLOCK_MANAGER_PORT = "4043"

In [16]:
# TODO: MAKE SURE TO RUN THIS IN ORDER TO HANDLE host.docker.internal in local host:
#  echo "127.0.0.1 host.docker.internal" | sudo tee -a /etc/hosts 

In [ ]:
AWS_BUNDLE = "com.amazonaws:aws-java-sdk-bundle:1.12.262"
HADOOP_AWS = "org.apache.hadoop:hadoop-aws:3.3.4"

spark = (
    SparkSession.builder
    .appName("data_cleansing")
    .master("spark://localhost:7077")          # driver is your notebook (client mode)
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.host", DRIVER_HOST)
    .config("spark.driver.port", DRIVER_PORT)
    .config("spark.blockManager.port", BLOCK_MANAGER_PORT)
    # ↓ Let Spark fetch & ship jars to executors
    .config("spark.jars.packages", f"{HADOOP_AWS},{AWS_BUNDLE}")
    # MINIO/S3A
    .config("spark.hadoop.fs.s3a.endpoint", "http://host.docker.internal:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    # Explicit creds provider (so the access/secret keys below are actually used)
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    # Failure settings
    .config("spark.hadoop.fs.s3a.connection.timeout", "10000")
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000")
    .config("spark.hadoop.fs.s3a.socket.timeout", "30000")
    .config("spark.hadoop.fs.s3a.attempts.maximum", "3")
    .config("spark.hadoop.fs.s3a.retry.limit", "2")
    .getOrCreate()
)

spark.version, spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()

:: loading settings :: url = jar:file:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/luisenrique/.ivy2/cache
The jars for the packages stored in: /Users/luisenrique/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ce7684c0-8c6d-4c1b-b8ad-f4804dd54f1a;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 92ms :: artifacts dl 3ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|d

('3.4.0', '3.3.4')

In [7]:
hadoop_config = spark._jsc.hadoopConfiguration()
print(hadoop_config.get("fs.s3a.endpoint"))
print(hadoop_config.get("fs.s3a.aws.credentials.provider"))
print(hadoop_config.get("fs.s3a.path.style.access"))
print(hadoop_config.get("fs.s3a.connection.ssl.enabled"))

http://host.docker.internal:9000
org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider
true
false


In [8]:
jvm = spark.sparkContext._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI("s3a://inventory/"), hadoop_config)
print(fs)
for s in fs.listStatus(jvm.org.apache.hadoop.fs.Path("s3a://inventory/")):
    print("->", s.getPath().toString())

25/11/07 19:13:45 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


S3AFileSystem{uri=s3a://inventory, workingDir=s3a://inventory/user/luisenrique, inputPolicy=normal, partSize=67108864, enableMultiObjectsDelete=true, maxKeys=5000, readAhead=65536, blockSize=33554432, multiPartThreshold=134217728, s3EncryptionAlgorithm='NONE', blockFactory=org.apache.hadoop.fs.s3a.S3ADataBlocks$DiskBlockFactory@4c7b6ba8, auditManager=Service NoopAuditManagerS3A in state NoopAuditManagerS3A: STARTED, metastore=NullMetadataStore, authoritativeStore=false, authoritativePath=[], useListV1=false, magicCommitter=true, boundedExecutor=BlockingThreadPoolExecutorService{SemaphoredDelegatingExecutor{permitCount=160, available=160, waiting=0}, activeCount=0}, unboundedExecutor=java.util.concurrent.ThreadPoolExecutor@2596ab20[Running, pool size = 0, active threads = 0, queued tasks = 0, completed tasks = 0], credentials=AWSCredentialProviderList[refcount= 1: [SimpleAWSCredentialsProvider], delegation tokens=disabled, DirectoryMarkerRetention{policy='delete'}, instrumentation {S3AI

In [9]:
spark._jsc.hadoopConfiguration().get("fs.s3a.impl")

'org.apache.hadoop.fs.s3a.S3AFileSystem'

In [10]:
spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()

# Driver JVM classpath (you'll see site-packages/pyspark/jars but no hadoop-aws/aws-sdk)
spark.sparkContext._jvm.java.lang.System.getProperty("java.class.path")

'/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.11/site-packages/pyspark/conf:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.11/site-packages/pyspark/jars/dropwizard-metrics-hadoop-metrics2-reporter-0.1.2.jar:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.11/site-packages/pyspark/jars/netty-handler-proxy-4.1.87.Final.jar:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.11/site-packages/pyspark/jars/logging-interceptor-3.12.12.jar:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.11/site-packages/pyspark/jars/threeten-extra-1.7.1.jar:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.11/site-packages/pyspark/jars/hive-shims-2.3.9.jar:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv/lib/python3.11/site-packages/pyspark/jars/spar

In [13]:
# TODO: REPLACE LOCAL CSV LOADING WITH A DISTRIBUTED STORAGE SOLUTION (E.G. S3)
df_raw = spark.read.csv("s3a://inventory/inventory_raw.csv", header=True, inferSchema=True)

In [15]:
df_raw.head(5)

[Row(source_row_id=1, ip='192.168.010.005', hostname='HOST01', fqdn=None, mac='AA-BB-CC-DD-EE-FF', owner='priya (platform) priya@corp.example.com', device_type='server', site='BLR Campus', notes='db host'),
 Row(source_row_id=2, ip='10.0.1.300', hostname='host-02', fqdn='host-02.local', mac='11-22-33-44-55-66', owner='ops', device_type=None, site='HQ Bldg 1', notes='edge gw?'),
 Row(source_row_id=3, ip='10.0.1', hostname='host03', fqdn=None, mac='aabb.ccdd.eeff', owner='jane@corp.example.com', device_type='switch', site='HQ-BUILDING-1', notes=None),
 Row(source_row_id=4, ip='10.0.1.1.2', hostname='printer-01', fqdn=None, mac='00:11:22:33:44:55', owner='Facilities', device_type='printer', site='HQ', notes=None),
 Row(source_row_id=5, ip='fe80::1%eth0', hostname='iot-cam01', fqdn=None, mac='00:aa:bb:cc:dd:ee', owner='sec', device_type='iot', site='Lab-1', notes='camera PoE on port 3')]